In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from typing import List
from typing import Tuple
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import ClusterCentroids
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_curve, auc

df = pd.read_csv("Credit_card.csv")
label_df = pd.read_csv('Credit_card_label.csv')
df.head(30)

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
#1.
from sklearn.preprocessing import LabelEncoder

class_labels = LabelEncoder()

df['Housing_type'] = class_labels.fit_transform(df['Housing_type'].values)
df['Type_Income'] = class_labels.fit_transform(df['Type_Income'].values)
df['GENDER'] = class_labels.fit_transform(df['GENDER'].values)
df['Car_Owner'] = class_labels.fit_transform(df['Car_Owner'].values)
df['Propert_Owner'] = class_labels.fit_transform(df['Propert_Owner'].values)
df['CHILDREN'] = class_labels.fit_transform(df['CHILDREN'].values)
df['EDUCATION'] = class_labels.fit_transform(df['EDUCATION'].values)
df['Marital_status'] = class_labels.fit_transform(df['Marital_status'].values)
df['Type_Occupation'] = class_labels.fit_transform(df['Type_Occupation'].values)
df['Annual_income'] = class_labels.fit_transform(df['Annual_income'].values)
df['Birthday_count'] = class_labels.fit_transform(df['Birthday_count'].values)
df['Employed_days'] = class_labels.fit_transform(df['Employed_days'].values)
df

# There wasn't really anything else I thought I could do so I just changed all the employed days to be positive.

In [ ]:
from statsmodels.formula.api import logit
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

y = df['Type_Occupation']
X = df[['Family_Members']]

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3, 
                                                    random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression

regression = LinearRegression()
regression.fit(X_train, y_train)

In [ ]:
y_pred = regression.predict(X_test) 
y_pred

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)

X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_resampled, y_resampled)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)

In [ ]:
#2
def apply_smoteenn(X:int, y:int):
    
    smoteenn = SMOTEENN(random_state=42)
    X_resampled, y_resampled = smoteenn.fit_resample(X, y)
    return X_resampled, y_resampled

In [ ]:
def train_logistic_regression(df_encoded: pd.DataFrame) -> float:
    X = df.drop(columns=['Car_Owner', 'Ind_ID'])
    y = df['Car_Owner']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train_resampled, y_train_resampled = apply_smoteenn(X_train, y_train)

    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_resampled, y_train_resampled)

    y_pred = log_reg.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    return accuracy, report
accuracy, report = train_logistic_regression(df)
print(accuracy)
print(report)

In [ ]:
def find_optimal_k(df: pd.DataFrame, 
                   target_col: str, 
                   k_range: Tuple[int, int] = (1, 21), 
                   test_size: float = 0.2, 
                   random_state: int = 42) -> float:

    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train_resampled, y_train_resampled = apply_smoteenn(X_train, y_train)
    best_k = 0
    best_score = 0
    for k in k_range:
        knn = KNeighborsClassifier(n_neighbors=((2*k)+1))
        knn.fit(X_train_resampled, y_train_resampled)
        score = knn.score(X_test, y_test)
        if score > best_score:
            best_k = k
            best_score = score

    knn = KNeighborsClassifier(n_neighbors=best_k)
    knn.fit(X_train_resampled, y_train_resampled)

    y_pred = knn.predict(X_test)
    y_probs = knn.predict_proba(X_test)[:, 1]
    
    test_accuracy = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)
    
    return best_k, test_accuracy, conf_matrix, class_report, y_test, y_probs

best_k, test_accuracy, conf_matrix, class_report, y_test, y_probs = find_optimal_k(df, target_col='Car_Owner')

print(f"Optimal k: {best_k}")
print(f"Test Accuracy: {test_accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

In [ ]:
#I chose to do oversampling on the fact that I thought it would simplify the imformation even though the precision was not that good. 
#There was not a large effect on my KNN and logostic regression.

In [ ]:
#3
def plot_roc_curve(y_test: np.ndarray, y_probs: np.ndarray) -> float:
    
    fpr, tpr, thresholds = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)
    
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc='lower right')
    plt.show()
    
    return roc_auc


best_k, test_accuracy, conf_matrix, class_report, y_test, y_probs = find_optimal_k(df, target_col='Car_Owner')

print(f"Optimal k: {best_k}")
print(f"Test Accuracy: {test_accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

roc_auc = plot_roc_curve(y_test, y_probs)
print(f"ROC AUC: {roc_auc}")

In [ ]:
#4
def train_evaluate_svm(df: pd.DataFrame, target_column: str = 'Car_Owner') -> None:
    X = df.drop(['Ind_ID', target_column], axis=1)
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    classifier = SVC(kernel='linear', random_state=42)
    classifier.fit(X_train, y_train)

    y_pred = classifier.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    classification_rep = classification_report(y_test, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(classification_rep)
train_evaluate_svm(df)
#The peformance was not that bad.

In [ ]:
#5
#Outlier detection is a way to detect data poits that are far away from the average. There are multiple methods that you can us for
#outlier detection inluding the Z-Method, IQR technique, and the clustering method.

In [ ]:
%pip install imblearn